# VIBE deployment in VS Code / WSL Ubuntu 20.04 with Docker Ubuntu 18.04

This notebook is intended to be opened in VS Code from the WSL Ubuntu 20.04 host.  
The notebook acts as the controller. The actual VIBE runtime is an Ubuntu 18.04 Docker container with Python 3.7, CUDA 10.1, PyTorch 1.4.0, and the official VIBE repository.

Do **not** use Ubuntu 18.04 as the main WSL distro. Use it only inside Docker.


In [ ]:
# CELL 1: Define project paths and a helper function to run shell commands from the WSL host

from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import json
import sys

PROJECT_DIR = Path.home() / "UA" / "TAVA" / "TAVA-Final-Practice" / "vibe-method"
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
DATASETS_DIR = PROJECT_DIR / "datasets"
TOOLS_DIR = PROJECT_DIR / "tools"

for d in [PROJECT_DIR, INPUT_DIR, OUTPUT_DIR, DATASETS_DIR, TOOLS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=PROJECT_DIR, check=True):
    """
    Run a shell command on the WSL host.
    The command output is printed immediately so this notebook can be used as a deployment log.
    """
    print(f"\n[HOST] {cmd}\n")
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result

print("Project directory:", PROJECT_DIR)
print("Input videos directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Tools directory:", TOOLS_DIR)


In [ ]:
# CELL 2: Write the Dockerfile and helper Python tools used by the VIBE container

dockerfile = r"""
FROM nvidia/cuda:10.1-cudnn7-devel-ubuntu18.04

ENV DEBIAN_FRONTEND=noninteractive
SHELL ["/bin/bash", "-lc"]

ENV CONDA_DIR=/opt/conda
ENV PATH=${CONDA_DIR}/bin:${PATH}
ENV PYOPENGL_PLATFORM=egl
ENV NVIDIA_VISIBLE_DEVICES=all
ENV NVIDIA_DRIVER_CAPABILITIES=compute,utility,graphics,video
ENV PIP_NO_CACHE_DIR=1

RUN apt-get update && apt-get install -y --no-install-recommends \
    git \
    wget \
    curl \
    ca-certificates \
    bzip2 \
    unzip \
    ffmpeg \
    build-essential \
    cmake \
    pkg-config \
    ninja-build \
    libgl1-mesa-glx \
    libglib2.0-0 \
    libsm6 \
    libxext6 \
    libxrender-dev \
    libegl1-mesa \
    libegl1-mesa-dev \
    libgles2-mesa-dev \
    libosmesa6-dev \
    libjpeg-dev \
    zlib1g-dev \
    libopenblas-dev \
    liblapack-dev \
    && rm -rf /var/lib/apt/lists/*

RUN wget -q https://repo.anaconda.com/miniconda/Miniconda3-py37_4.12.0-Linux-x86_64.sh -O /tmp/miniconda.sh \
    && bash /tmp/miniconda.sh -b -p ${CONDA_DIR} \
    && rm /tmp/miniconda.sh \
    && conda config --set auto_update_conda false \
    && conda clean -afy

RUN python -m pip install --upgrade pip==23.0.1 setuptools==59.8.0 wheel==0.38.4

WORKDIR /workspace

RUN git clone https://github.com/mkocabas/VIBE.git /workspace/VIBE

WORKDIR /workspace/VIBE

# Official VIBE install script pins torch==1.4.0, torchvision==0.5.0, numpy==1.17.5.
# We install them explicitly before the repository requirements to reduce dependency drift.
RUN python -m pip install numpy==1.17.5
RUN python -m pip install torch==1.4.0 torchvision==0.5.0
RUN python -m pip install git+https://github.com/giacaglia/pytube.git --upgrade
RUN python -m pip install -r requirements.txt

RUN mkdir -p /workspace/input /workspace/output /workspace/datasets /workspace/tools

CMD ["/bin/bash"]
""".strip()

inspect_script = r"""
# Inspect a VIBE output .pkl file.

import argparse
from pathlib import Path

import joblib
import numpy as np

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("pkl_path", type=str, help="Path to vibe_output.pkl")
    args = parser.parse_args()

    pkl_path = Path(args.pkl_path)
    if not pkl_path.exists():
        raise FileNotFoundError(pkl_path)

    output = joblib.load(str(pkl_path))
    print("Loaded:", pkl_path)
    print("Track IDs:", list(output.keys()))

    for person_id, person_data in output.items():
        print("\n==============================")
        print(f"Person / track ID: {person_id}")
        for key, value in person_data.items():
            if hasattr(value, "shape"):
                print(f"{key:20s} shape={value.shape} dtype={getattr(value, 'dtype', None)}")
            else:
                print(f"{key:20s} type={type(value)}")

        if "joints3d" in person_data:
            joints = np.asarray(person_data["joints3d"])
            print("joints3d min/max:", float(np.nanmin(joints)), float(np.nanmax(joints)))

if __name__ == "__main__":
    main()
""".strip()

metrics_script = r"""
# Generic metric utilities for SMPL/HMR evaluation.
# Use these functions after you have matched VIBE predictions with ground-truth frames/person IDs.

from __future__ import annotations

import argparse
import numpy as np


def _to_numpy(x):
    return np.asarray(x, dtype=np.float64)


def mpjpe(pred_joints, gt_joints, valid_mask=None):
    pred = _to_numpy(pred_joints)
    gt = _to_numpy(gt_joints)
    err = np.linalg.norm(pred - gt, axis=-1)
    if valid_mask is not None:
        err = err[np.asarray(valid_mask).astype(bool)]
    return float(np.mean(err))


def pck(pred_joints, gt_joints, threshold=0.15, valid_mask=None):
    pred = _to_numpy(pred_joints)
    gt = _to_numpy(gt_joints)
    err = np.linalg.norm(pred - gt, axis=-1)
    correct = err < threshold
    if valid_mask is not None:
        correct = correct[np.asarray(valid_mask).astype(bool)]
    return float(np.mean(correct))


def compute_similarity_transform_batch(pred, gt):
    pred = _to_numpy(pred)
    gt = _to_numpy(gt)
    assert pred.shape == gt.shape and pred.ndim == 3 and pred.shape[-1] == 3

    aligned = np.zeros_like(pred)

    for i in range(pred.shape[0]):
        X = pred[i].T
        Y = gt[i].T

        muX = X.mean(axis=1, keepdims=True)
        muY = Y.mean(axis=1, keepdims=True)
        X0 = X - muX
        Y0 = Y - muY

        normX = np.sqrt((X0 ** 2).sum())
        normY = np.sqrt((Y0 ** 2).sum())

        if normX < 1e-8 or normY < 1e-8:
            aligned[i] = pred[i]
            continue

        X0 /= normX
        Y0 /= normY

        H = X0 @ Y0.T
        U, s, Vt = np.linalg.svd(H)
        V = Vt.T
        R = V @ U.T

        if np.linalg.det(R) < 0:
            V[:, -1] *= -1
            s[-1] *= -1
            R = V @ U.T

        scale = (s.sum() * normY) / normX
        t = muY - scale * R @ muX
        X_aligned = scale * R @ X + t
        aligned[i] = X_aligned.T

    return aligned


def pa_mpjpe(pred_joints, gt_joints):
    pred = _to_numpy(pred_joints)
    gt = _to_numpy(gt_joints)
    aligned = compute_similarity_transform_batch(pred, gt)
    return mpjpe(aligned, gt)


def pve(pred_vertices, gt_vertices):
    pred = _to_numpy(pred_vertices)
    gt = _to_numpy(gt_vertices)
    return float(np.mean(np.linalg.norm(pred - gt, axis=-1)))


def beta_l2(pred_betas, gt_betas):
    pred = _to_numpy(pred_betas)
    gt = _to_numpy(gt_betas)
    return float(np.mean(np.linalg.norm(pred - gt, axis=-1)))


def acceleration_error(pred_joints, gt_joints=None):
    pred = _to_numpy(pred_joints)
    pred_acc = pred[2:] - 2.0 * pred[1:-1] + pred[:-2]

    if gt_joints is None:
        return float(np.mean(np.linalg.norm(pred_acc, axis=-1)))

    gt = _to_numpy(gt_joints)
    gt_acc = gt[2:] - 2.0 * gt[1:-1] + gt[:-2]
    return float(np.mean(np.linalg.norm(pred_acc - gt_acc, axis=-1)))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pred_npz", required=True, help="NPZ with prediction arrays")
    parser.add_argument("--gt_npz", required=True, help="NPZ with ground-truth arrays")
    parser.add_argument("--joint_key", default="joints3d", help="Array key for joints")
    parser.add_argument("--vertex_key", default="verts", help="Array key for vertices")
    parser.add_argument("--beta_key", default="betas", help="Array key for SMPL betas")
    args = parser.parse_args()

    pred = np.load(args.pred_npz)
    gt = np.load(args.gt_npz)

    if args.joint_key in pred and args.joint_key in gt:
        print("MPJPE:", mpjpe(pred[args.joint_key], gt[args.joint_key]))
        print("PA-MPJPE:", pa_mpjpe(pred[args.joint_key], gt[args.joint_key]))
        print("Acceleration error:", acceleration_error(pred[args.joint_key], gt[args.joint_key]))

    if args.vertex_key in pred and args.vertex_key in gt:
        print("PVE:", pve(pred[args.vertex_key], gt[args.vertex_key]))

    if args.beta_key in pred and args.beta_key in gt:
        print("Beta L2:", beta_l2(pred[args.beta_key], gt[args.beta_key]))

if __name__ == "__main__":
    main()
""".strip()

readme = r"""
# VIBE deployment bundle

Open `VIBE_Docker_Deployment.ipynb` in VS Code from the WSL Ubuntu 20.04 host.

The notebook:
1. Creates a Dockerfile for an Ubuntu 18.04 + CUDA 10.1 + Python 3.7 VIBE runtime.
2. Builds the Docker image.
3. Starts a GPU-enabled container.
4. Downloads VIBE pretrained data with the official script.
5. Runs VIBE on the official sample video or your own video.
6. Inspects `vibe_output.pkl`.
7. Provides metric utilities for MPJPE, PA-MPJPE, PVE, beta error, PCK, and acceleration error.
""".strip()

(PROJECT_DIR / "Dockerfile").write_text(dockerfile + "\n", encoding="utf-8")
(TOOLS_DIR / "inspect_vibe_output.py").write_text(inspect_script + "\n", encoding="utf-8")
(TOOLS_DIR / "vibe_metrics.py").write_text(metrics_script + "\n", encoding="utf-8")
(PROJECT_DIR / "README.md").write_text(readme + "\n", encoding="utf-8")

print("Wrote:")
for p in [PROJECT_DIR / "Dockerfile", TOOLS_DIR / "inspect_vibe_output.py", TOOLS_DIR / "vibe_metrics.py", PROJECT_DIR / "README.md"]:
    print(" -", p)


In [ ]:
# CELL 3: Check that Docker is available from WSL and that the current user can call it

run("docker version", check=True)

# If this fails with permission denied, run this in a WSL terminal, then restart WSL:
# sudo usermod -aG docker $USER
# exit
# wsl --shutdown


In [ ]:
# CELL 4: Validate NVIDIA GPU access inside Docker

# This is the official Docker/NVIDIA-style validation step.
# If you use Docker Desktop, make sure Docker Desktop is running and WSL integration is enabled.
# If you use Docker Engine inside WSL, make sure nvidia-container-toolkit is installed.

run("docker run --rm --gpus all nvcr.io/nvidia/k8s/cuda-sample:nbody nbody -gpu -benchmark", check=True)


In [ ]:
# CELL 5: Build the VIBE Docker image

# This can take a while because it installs the full old VIBE dependency stack.
# The image tag is intentionally explicit: Ubuntu 18.04 + Python 3.7 + CUDA 10.1.
run("docker build -t vibe-ubuntu18:py37-cu101 .", check=True)


In [ ]:
# CELL 6: Start a persistent VIBE development container with mounted input/output folders

run("docker rm -f vibe-dev || true", check=False)

docker_run_cmd = f"""
docker run -dit \
  --name vibe-dev \
  --gpus all \
  --ipc=host \
  -e PYOPENGL_PLATFORM=egl \
  -e NVIDIA_DRIVER_CAPABILITIES=compute,utility,graphics,video \
  -v "{INPUT_DIR}:/workspace/input" \
  -v "{OUTPUT_DIR}:/workspace/output" \
  -v "{DATASETS_DIR}:/workspace/datasets" \
  -v "{TOOLS_DIR}:/workspace/tools" \
  vibe-ubuntu18:py37-cu101 \
  bash
""".strip()

run(docker_run_cmd, check=True)
run("docker ps --filter name=vibe-dev", check=True)


In [ ]:
# CELL 7: Smoke-test the VIBE runtime inside the Docker container

runtime_test = r"""
set -e
python --version
python - <<'PY'
import torch
import torchvision
import numpy as np
import cv2
import pyrender

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)
print("cv2:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
print("pyrender import: OK")
PY
"""

run(f"docker exec vibe-dev bash -lc {shlex.quote(runtime_test)}", check=True)


In [ ]:
# CELL 8: Download VIBE pretrained data, SMPL-related data, sample video, and YOLO weights

# The official VIBE script downloads data/vibe_data.zip, unzips it,
# moves sample_video.mp4 to the repository root, and moves yolov3.weights to ~/.torch/models.
prepare_cmd = r"""
set -e
cd /workspace/VIBE

if [ -d data/vibe_data ] && [ -f sample_video.mp4 ]; then
    echo "VIBE data already exists. Skipping scripts/prepare_data.sh"
else
    source scripts/prepare_data.sh
fi

echo "Repository sample video:"
ls -lh /workspace/VIBE/sample_video.mp4 || true

echo "VIBE data directory:"
find /workspace/VIBE/data/vibe_data -maxdepth 2 -type f | head -50
"""

run(f"docker exec vibe-dev bash -lc {shlex.quote(prepare_cmd)}", check=True)


In [ ]:
# CELL 9: Prepare a test input video

# Option A: use VIBE's official sample video for the first deployment test.
run("docker exec vibe-dev bash -lc 'cp /workspace/VIBE/sample_video.mp4 /workspace/input/sample_video.mp4 && ls -lh /workspace/input/sample_video.mp4'", check=True)

# Option B: for your own video, copy it from Windows into this WSL folder:
#   /home/<your-user>/UA/TAVA/TAVA-Final-Practice/vibe-method/input/
# Then change VIDEO_NAME below.
VIDEO_NAME = "sample_video.mp4"
print("Selected video:", VIDEO_NAME)


In [ ]:
# CELL 10: Run VIBE inference without rendering first

# This is the recommended first successful run.
# Do not use --display inside a headless Docker/VS Code notebook.
# Reduce --tracker_batch_size and --vibe_batch_size if you get CUDA out-of-memory errors.

video_path = f"/workspace/input/{VIDEO_NAME}"

demo_cmd = f"""
set -e
cd /workspace/VIBE
python demo.py \
  --vid_file {shlex.quote(video_path)} \
  --output_folder /workspace/output \
  --tracking_method bbox \
  --detector yolo \
  --tracker_batch_size 4 \
  --vibe_batch_size 64 \
  --no_render
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(demo_cmd)}", check=True)


In [ ]:
# CELL 11: Inspect the VIBE output pickle

# The output folder is named after the input video without the .mp4 extension.
video_stem = Path(VIDEO_NAME).stem
vibe_output_pkl = f"/workspace/output/{video_stem}/vibe_output.pkl"

inspect_cmd = f"""
set -e
python /workspace/tools/inspect_vibe_output.py {shlex.quote(vibe_output_pkl)}
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(inspect_cmd)}", check=True)


In [ ]:
# CELL 12: Run VIBE again with rendering enabled

# Use this after CELL 10 works.
# Rendering uses pyrender/EGL and may be slower than inference.
# If rendering fails on your GPU/driver setup, keep --no_render for quantitative evaluation.

render_cmd = f"""
set -e
cd /workspace/VIBE
python demo.py \
  --vid_file {shlex.quote(video_path)} \
  --output_folder /workspace/output \
  --tracking_method bbox \
  --detector yolo \
  --tracker_batch_size 4 \
  --vibe_batch_size 64 \
  --sideview
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(render_cmd)}", check=False)


In [ ]:
# CELL 13: Compute a temporal smoothness proxy from the VIBE output

# This does not need ground truth.
# It computes the mean magnitude of predicted joint acceleration per track.
# Lower values generally indicate smoother predictions, but this is not a substitute for GT acceleration error.

smoothness_cmd = f"""
set -e
python - <<'PY'
import joblib
import numpy as np
import sys
sys.path.insert(0, "/workspace/tools")
from vibe_metrics import acceleration_error

pkl_path = "{vibe_output_pkl}"
data = joblib.load(pkl_path)

print("Temporal smoothness proxy from:", pkl_path)
for pid, person_data in data.items():
    joints = np.asarray(person_data["joints3d"])
    if joints.shape[0] < 3:
        print(f"track {{pid}}: not enough frames")
        continue
    score = acceleration_error(joints, gt_joints=None)
    print(f"track {{pid}}: predicted acceleration magnitude = {{score:.6f}}")
PY
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(smoothness_cmd)}", check=True)


In [ ]:
# CELL 14: Export one VIBE track to NPZ for later quantitative evaluation

# This cell creates a simple NPZ file from the first detected person track.
# You can then align/match it with a dataset-specific ground-truth NPZ.

export_cmd = f"""
set -e
python - <<'PY'
import joblib
import numpy as np
from pathlib import Path

pkl_path = Path("{vibe_output_pkl}")
out_path = pkl_path.parent / "first_track_prediction.npz"

data = joblib.load(str(pkl_path))
first_pid = sorted(data.keys())[0]
track = data[first_pid]

np.savez_compressed(
    out_path,
    pred_cam=track["pred_cam"],
    orig_cam=track["orig_cam"],
    verts=track["verts"],
    pose=track["pose"],
    betas=track["betas"],
    joints3d=track["joints3d"],
    bboxes=track["bboxes"],
    frame_ids=track["frame_ids"],
)

print("Exported:", out_path)
print("Track ID:", first_pid)
print("Frames:", track["frame_ids"].shape)
PY
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(export_cmd)}", check=True)


In [ ]:
# CELL 15: Evaluate against a prepared ground-truth NPZ file

# This cell is a template for real dataset evaluation.
# You must create a GT NPZ with matched arrays and the same coordinate system:
#   gt_joints3d: shape (T, J, 3)
#   gt_verts:    shape (T, 6890, 3), optional
#   gt_betas:    shape (T, 10), optional
#
# Then rename keys or create a compatible GT file with:
#   joints3d, verts, betas
#
# Example expected files:
#   /workspace/output/sample_video/first_track_prediction.npz
#   /workspace/datasets/your_dataset/matched_ground_truth.npz

pred_npz = f"/workspace/output/{video_stem}/first_track_prediction.npz"
gt_npz = "/workspace/datasets/your_dataset/matched_ground_truth.npz"

eval_cmd = f"""
set -e
if [ ! -f {shlex.quote(gt_npz)} ]; then
    echo "Ground-truth NPZ not found: {gt_npz}"
    echo "Create a matched GT file before running quantitative metrics."
    exit 0
fi

python /workspace/tools/vibe_metrics.py \
  --pred_npz {shlex.quote(pred_npz)} \
  --gt_npz {shlex.quote(gt_npz)} \
  --joint_key joints3d \
  --vertex_key verts \
  --beta_key betas
""".strip()

run(f"docker exec vibe-dev bash -lc {shlex.quote(eval_cmd)}", check=True)


In [ ]:
# CELL 16: Optional cleanup commands

# Stop the container:
# run("docker stop vibe-dev", check=False)

# Remove the container:
# run("docker rm -f vibe-dev", check=False)

# Remove the image:
# run("docker rmi vibe-ubuntu18:py37-cu101", check=False)

print("Cleanup cell loaded. Uncomment the commands above only when you want to clean the environment.")
